In [ ]:
import pandas as pd 

df=pd.read_csv("../data/SMSSpamCollection",sep="\t",header=None,names=["label","message"])


print(df.head())

In [ ]:
df["message_length"] = df["message"].str.len()

print(df.head())

In [ ]:
df[df.duplicated(keep=False)].sort_values("label").head(20)

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

In [ ]:
print(df.groupby("label")["message_length"].describe())
df.groupby("label")["message_length"].mean()
df.groupby("label")["message_length"].median()

In [ ]:
import matplotlib.pyplot as plt

df.boxplot(column="message_length", by="label")

plt.title("Message Length by Class")
plt.suptitle("")
plt.xlabel("Label")
plt.ylabel("Message Length")
plt.show()

In [22]:
df["label_num"]=df["label"].map({
    "ham":0,
    "spam":1
})
df.head()

,label,message,message_length,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",111,0
1,ham,Ok lar... Joking wif u oni...,29,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155,1
3,ham,U dun say so early hor... U c already then say...,49,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",61,0


In [ ]:
x=df["message"]
y=df["label_num"]
print("X shape:", x.shape)
print("y shape:", y.shape)


In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(x_train))
print("Testing samples:", len(x_test))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=5000
)

x_train_tfidf = tfidf.fit_transform(x_train)
x_test_tfidf = tfidf.transform(x_test)

print("Training shape:", x_train_tfidf.shape)
print("Testing shape:", x_test_tfidf.shape)



In [ ]:

print("Training samples:", x_train_tfidf.shape[0])
print("Training features:", x_train_tfidf.shape[1])

print("Testing samples:", x_test_tfidf.shape[0])
print("Testing features:", x_test_tfidf.shape[1])

In [ ]:
from sklearn.linear_model import LogisticRegression

spam_lr = LogisticRegression(max_iter=1000)

spam_lr.fit(x_train_tfidf, y_train)

print("Spam model training completed!")

Spam model training completed!


In [ ]:
y_pred=spam_lr.predict(x_test_tfidf)

print(y_pred[:20])

In [19]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9564796905222437

Classification Report:
              precision    recall  f1-score   support

         Ham       0.95      1.00      0.98       903
        Spam       0.98      0.67      0.80       131

    accuracy                           0.96      1034
   macro avg       0.97      0.83      0.89      1034
weighted avg       0.96      0.96      0.95      1034


Confusion Matrix:
[[901   2]
 [ 43  88]]


In [ ]:
from sklearn.svm import LinearSVC

spam_model = LinearSVC()

spam_model.fit(x_train_tfidf, y_train)

y_pred_svc = spam_model.predict(x_test_tfidf)

print("LinearSVC Accuracy:", accuracy_score(y_test, y_pred_svc))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_svc,
    target_names=["Ham", "Spam"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_svc))

LinearSVC Accuracy: 0.9796905222437138

Classification Report:
              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99       903
        Spam       0.98      0.85      0.91       131

    accuracy                           0.98      1034
   macro avg       0.98      0.93      0.95      1034
weighted avg       0.98      0.98      0.98      1034


Confusion Matrix:
[[901   2]
 [ 19 112]]


In [ ]:
from sklearn.pipeline import Pipeline

spam_pipe=Pipeline([
    ("tfidf",TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )),
    ("model",spam_model)
])

spam_pipe.fit(x_train,y_train)
pre=spam_pipe.predict(x_test)
print(pre[:10])

[0 0 0 0 0 0 0 0 0 0]


In [ ]:
message=["Congratulations! You won a free iPhone. Click now!",
    "Hey, are we meeting tomorrow at college?"]

pred=spam_pipe.predict(message)

print(pred)

[1 0]


In [23]:
import joblib as jb 

jb.dump(spam_pipe,"spam_pred.pkl")

['spam_pred.pkl']